<!-- SOLUTION ONLY -->
> **🔑 SOLUTION — instructors only.** Do not distribute to students before the session.

# TP3a — A global ocean model: NEMO at 1°

**Prerequisites**: Course 2 (numerical models), TP1 and TP2.

In TP1 and TP2 you worked with **observations** and with a **reanalysis**. Today you open the output of a **global ocean model**, NEMO, and right after (TP3b) a short look at a **regional** one, CROCO. Its file looks a lot like the observation files, so the same xarray recipes work — with three traps to know.
## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete**. Everything else (imports, helper functions, plot formatting) is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like — use it to self-correct before moving on.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click. Try for a few minutes before opening them!
- **🚀 Advanced track** blocks are optional: if you finish an exercise before the others, try them while the instructor helps the rest of the group.

> **⚠️ Before you start: work on your own copy.** In the JupyterLab file browser, right-click this notebook → *Duplicate*, then rename the copy to `03a_nemo_EN_mine.ipynb` and open it. Work only in your `_mine` copy, so the original stays clean and you can always go back to it.
## Reminder from Course 2

| Term | Meaning |
|---|---|
| **Numerical model** | a program that solves the equations of the ocean (currents, temperature, salinity) on a grid, step by step in time |
| **Grid cell, resolution** | the model computes **one value per cell**: anything smaller than a few cells cannot be represented |
| **Vertical levels** | the layers of the model: **z-levels** (fixed depths, the same everywhere) or **sigma levels** (they follow the sea floor) |
| **Global / regional model** | the whole ocean at coarse resolution / one region at fine resolution, fed at its **open boundaries** by a larger model |
| **Forcing** | what drives the model from outside: wind, heat and fresh water from the atmosphere (and, for a regional model, the ocean at its boundaries) |
| **Free run / reanalysis** | a free run is only driven by its forcing; a reanalysis (GLORYS12, TP2) is also **corrected by observations** |

## Today's study area: Costa Rica seen by a global model
Same region and same story as in TP1 and TP2: the **Pacific coast of Costa Rica**, the **Papagayo jet** that cools the sea in winter, and offshore the **Costa Rica Dome**, where the thermocline comes close to the surface. A global model sees them **from far away**: it covers the whole Pacific, where El Niño is born (TP2b), but with cells of \~110 km. The run used here is **NEMO 4.2 coupled with PISCES** (biogeochemistry), forced by an atmospheric reanalysis: monthly means, surface fields for 2006–2025 and 3D fields for 2006.

| **Model** | **TP3a — NEMO** | **TP3b — CROCO** |
|---|---|---|
| Domain | **global** | **regional**: North-East Tropical Pacific |
| Horizontal grid | 1° (≈ 110 km) | ≈ 5.5 km |
| Vertical levels | **z-levels**: fixed depths, the same everywhere | **sigma levels**: they follow the sea floor |

- Section 1: the file of a global model, and its three traps (longitudes 0–360, a model calendar, the name of the vertical axis)
- Part A: the globe, a zoom on Costa Rica (what a 1° grid can and cannot show near the coast), then back to the whole Pacific: **El Niño, seen from where it is born**
- Part B: z-levels, a section across the Costa Rica Dome, a map at 100 m, then **one section across the whole Pacific**
- Section D: a recap table, completed for CROCO at the end of TP3b

## Contents
1. [Setup — imports and paths](#0-setup)
2. [Anatomy of a global model file](#1-anatomy)
3. **[Part A — From the globe to Costa Rica, and back](#part-a)**
   - [A.1 A global map](#a-1)
   - [A.2 Zoom on Costa Rica: the 0–360 trap](#a-2)
   - [A.3 Our box is not alone: El Niño seen from the whole Pacific](#a-3)
   - [🚀 Advanced track A — longitudes from −180 to 180](#advanced-a)
4. **[Part B — z-levels: the vertical grid](#part-b)**
   - [B.1 Where are the model levels?](#b-1)
   - [B.2 A vertical section across the Costa Rica Dome](#b-2)
   - [B.3 A map at 100 m](#b-3)
   - [B.4 The whole Pacific in one section: the equator](#b-4)
   - [🚀 Advanced track B — the depth of the thermocline over the whole tropical Pacific](#advanced-b)
5. [C. 🐞 Debugging exercise](#c-debugging)
6. [D. Recap: what to remember for TP3b](#d-recap)

---
<a id="0-setup"></a>
## 0. Setup — imports and paths

Run this cell as is. `base_map()` (land, coastlines and grid on a map) comes from `croco_helpers.py`, next to this notebook: keep your `_mine` copy **in this folder**.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

# Data already on Kabre (read-only)
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

NEMO_2D = f"{OBS_DIR}/MODEL_DS/NEMO/2D_Nemo4.2-PISCES_A_1_gr_20060101_20251231.nc"         # surface, monthly, 2006-2025
NEMO_3D = f"{OBS_DIR}/MODEL_DS/NEMO/3D_Nemo4.2-PISCES_A_1_gr_20060101_20251231_extr2006.nc"  # 3D, monthly, 2006

# Study box of Session 2 (Pacific and Caribbean sides of Costa Rica)
LON_MIN, LON_MAX = -92, -77
LAT_MIN, LAT_MAX = 4, 14

from croco_helpers import base_map

!ls -lh {OBS_DIR}/MODEL_DS/NEMO

**✅ Check**: the `ls` shows two files: the 2D file (\~710 MB) and the 3D file (\~1.3 GB). Big files, but nothing is loaded yet: xarray reads only the part you select.

---
<a id="1-anatomy"></a>
## 1. Anatomy of a global model file

Open the two files the usual way, as in TP1 and TP2.

In [ ]:
ds2 = xr.open_dataset(NEMO_2D)      # surface fields
ds3 = xr.open_dataset(NEMO_3D)      # 3D fields
ds3

**🎯 Your turn**: answer these questions from the output above (click on the 📄 and 🛢 icons to see attributes and values), then complete the cell to check your answers.
1. What are the dimensions of `thetao` (temperature)? What is the name of the vertical dimension, and of the time dimension?
2. What are the first and last values of `lon`? Of `lat`?
3. How many vertical levels are there? What are the first and last depths?
4. Which **calendar** does the time axis use (attributes of `time_counter`)?

In [ ]:
print("thetao dims :", ds3["thetao"].dims)
### STUDENT: print("lon first / last:", ..., ...)          # TODO
### STUDENT: print("lat first / last:", ..., ...)          # TODO
### STUDENT: print("levels:", ..., "| first / last depth (m):", ..., ...)   # TODO
### STUDENT: print("calendar:", ...)                       # TODO
### BEGIN SOLUTION
print("lon first / last:", float(ds3["lon"][0]), float(ds3["lon"][-1]))
print("lat first / last:", float(ds3["lat"][0]), float(ds3["lat"][-1]))
print("levels:", ds3.sizes["deptht"], "| first / last depth (m):", float(ds3["deptht"][0]), float(ds3["deptht"][-1]))
print("calendar:", ds3["time_counter"].encoding["calendar"])
### END SOLUTION
print("dates of the 2D file:", ds2["time_counter"].values[0], "...", ds2["time_counter"].values[-1])
### HINT: `float(ds3["lon"][0])` and `float(ds3["lon"][-1])`; the same for `lat`.
### HINT: Number of levels: `ds3.sizes["deptht"]`. Once decoded by xarray, the calendar is in `ds3["time_counter"].encoding["calendar"]`.

**✅ Check** — and what it means:

1. `thetao` is `(time_counter, deptht, lat, lon)`. Same structure as GLORYS in TP2a (`time, depth, latitude, longitude`), with **other names**: every model chooses its own (`time_counter`, `deptht` in NEMO). Always read the header first.
2. `lon` goes from **0 to 359**, `lat` from **−89.5 to 89.5**: a regular **1° grid**, with 1D coordinates like the observations. ⚠️ Longitudes are **east of Greenwich, 0 to 360**: Costa Rica is not at −85° but at **275°E** (= −85 + 360). Trap n°1, Part A.
3. **75 levels**, from **0.5 m to 5902 m**. `deptht` is a dimension with **fixed values**: level 10 is at the same depth everywhere on the globe. These are **z-levels** (Part B).
4. Calendar **`365_day`**: the model has **no 29 February**, every year has 365 days. xarray turns the dates into special `cftime` objects (`DatetimeNoLeap`) instead of the usual dates. Selecting a month works as usual (`.sel(time_counter="2023-12")`), but comparing day by day with an observation file needs care. Trap n°2.

💬 One more thing in the global attributes (`history`): the model was not run on this grid. NEMO runs on a curvilinear grid called **ORCA1** (with refined cells near the equator); the producer **regridded** the outputs to this regular 1° grid (`cdo remapdis,r360x180`) to make them easy to use. Keep it in mind for A.2.

---
<a id="part-a"></a>
# Part A — From the globe to Costa Rica, and back

<a id="a-1"></a>
## A.1 A global map

A global model covers the whole ocean: let's look at it all.

**🎯 Your turn**: select the sea surface temperature `tos` of **December 2023** in the 2D file. The cell maps it on the globe.

In [ ]:
### STUDENT: # TODO: tos for December 2023 (hint: .sel(time_counter="YYYY-MM")), then .squeeze() to drop the time dimension of size 1
### STUDENT: sst_dec = ...
### BEGIN SOLUTION
sst_dec = ds2["tos"].sel(time_counter="2023-12").squeeze()
### END SOLUTION
print(dict(sst_dec.sizes))

fig, ax = plt.subplots(figsize=(13, 6.5), subplot_kw={"projection": ccrs.Robinson(central_longitude=-150)})
sst_dec.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="jet", vmin=-2, vmax=31,
             cbar_kwargs={"label": "SST (°C)", "shrink": 0.7})
ax.plot([LON_MIN, LON_MAX, LON_MAX, LON_MIN, LON_MIN], [LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX, LAT_MIN],
        color="white", linewidth=2, transform=ccrs.PlateCarree())       # our study box
ax.add_feature(cfeature.LAND, facecolor="#d4c9a8")
ax.coastlines(linewidth=0.5)
ax.set_title("NEMO 1° — SST, December 2023 (white: our Costa Rica box)")
plt.show()
### HINT: `ds2["tos"].sel(time_counter="2023-12")` keeps one month; `.squeeze()` removes the `time_counter` dimension of size 1.

**✅ Check**: `sst_dec` is `{'lat': 180, 'lon': 360}`: one value per 1° cell, **64 800 cells** for the whole globe. Our box (magenta) is tiny on this map: about **150 cells** of the model.

**🎯 Read the globe** (with your partner): find on the map
1. the **warm pool** of the western Pacific and Indian Ocean (> 29 °C);
2. the **cold water** of the high latitudes (−2 to 0 °C near the sea ice), and the Southern Ocean, which goes all around the globe with no coast to stop it;
3. the **cold tongues along the eastern coasts**: off Peru–Chile, California, Namibia (Benguela): wind-driven **coastal upwelling**, the same mechanism as the Papagayo jet, on a larger scale;
4. the **eastern equatorial Pacific**: usually a cold tongue along the equator, but **not in December 2023**, in the middle of the **2023-24 El Niño** (TP2b). Keep this in mind for A.3.

💬 None of these can be seen from inside our box. A global model has **no open boundary**: the only thing it receives from outside is the atmosphere (wind, heat, fresh water) at the surface.

---
<a id="a-2"></a>
## A.2 Zoom on Costa Rica: the 0–360 trap

First try the "usual" selection, with the box of Session 2:

In [ ]:
zoom_wrong = sst_dec.sel(lon=slice(LON_MIN, LON_MAX), lat=slice(LAT_MIN, LAT_MAX))
print(dict(zoom_wrong.sizes))

**No error, but `lon: 0`**: the selection is **empty**. Longitudes −92 to −77 do not exist in a 0–360 file: xarray silently returns nothing (the same kind of silent trap as the latitude order in TP1a).

**🎯 Your turn**: select the box correctly, by converting the longitudes to the 0–360 convention (add 360). Then count the **ocean cells** of the box (`.count()` counts the values that are not NaN).

In [ ]:
### STUDENT: # TODO: the box in the 0-360 convention
### STUDENT: zoom = sst_dec.sel(lon=slice(..., ...), lat=slice(LAT_MIN, LAT_MAX))
### STUDENT: print(dict(zoom.sizes), "| ocean cells:", ...)
### BEGIN SOLUTION
zoom = sst_dec.sel(lon=slice(LON_MIN + 360, LON_MAX + 360), lat=slice(LAT_MIN, LAT_MAX))
print(dict(zoom.sizes), "| ocean cells:", int(zoom.count()))
### END SOLUTION

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
zoom.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="jet", vmin=26, vmax=30.5,
          edgecolors="0.5", linewidth=0.3,                     # draw the edge of each model cell
          cbar_kwargs={"label": "SST (°C)", "shrink": 0.7})
base_map(ax, "NEMO 1° — SST, December 2023: one square = one model cell",
         extent=[LON_MIN - 0.5, LON_MAX + 0.5, LAT_MIN - 0.5, LAT_MAX + 0.5], land=False)   # coastline only
plt.show()
### HINT: −92 + 360 = 268: `lon=slice(LON_MIN + 360, LON_MAX + 360)`.
### HINT: Ocean cells: `int(zoom.count())`.

**✅ Check**: `{'lat': 10, 'lon': 16}` and **153 ocean cells** out of 160, for a box of 15° × 10°. Look at the map closely (here only the coastline is drawn, not the land):
- each square is **one model cell, \~110 km wide**: Costa Rica is only **3 to 4 cells** across. The Gulf of Nicoya, the Gulf of Papagayo and the shelf are **smaller than one cell**: the model cannot represent them;
- the coast is a **staircase**, and many cells over Costa Rica, Nicaragua and Panama have a temperature, as if they were ocean: the Pacific and the Caribbean even seem to touch. That's the **regridding** of section 1: cells that are partly on land received values from their ocean neighbours. In the model itself (ORCA1 grid) the isthmus is closed. Always draw the coastline on top of a model field;
- the **cold patch off the Gulf of Papagayo** (\~10-11°N, 87-90°W, down to **\~26.5 °C**) is there, but drawn with a handful of cells.

💬 A regional model at \~5.5 km (TP3b) puts **about 400 grid points** in each of these squares (20 × 20). That's the first difference between global and regional: **horizontal resolution near the coast**.

---
<a id="a-3"></a>
## A.3 Our box is not alone: El Niño seen from the whole Pacific

In A.2 our box was 153 cells, with the coast drawn as a staircase. Seen this way, a 1° model looks like a poor tool for Costa Rica. But it can do something that **no regional model and no local observation** can do: show **where** an anomaly comes from.

In TP2b you found that 2023 was unusually warm at Costa Rica, because of El Niño. From inside the box you only see the **consequence**. Let's look at the whole globe.

**Step 1 — the anomaly map.** The anomaly of December 2023 is its SST minus the **mean of all the Decembers** of the file (2006–2025, 20 Decembers): what was unusual in that month.

**🎯 Your turn**: compute the December climatology and the anomaly of December 2023. The cell maps it, with our box (magenta) and the **Niño 3.4** box (black: 170°W–120°W, 5°S–5°N), the box used worldwide to measure El Niño.

In [ ]:
# Nino 3.4 box: 170°W-120°W, 5°S-5°N. In this 0-360 file: 190°E to 240°E (no trap here, all positive!)
NINO_LON_MIN, NINO_LON_MAX = 190, 240
NINO_LAT_MIN, NINO_LAT_MAX = -5, 5

tos = ds2["tos"]
### STUDENT: # TODO: mean of all the Decembers (keep the months where tos["time_counter"].dt.month == 12), then the anomaly of December 2023
### STUDENT: dec_clim = ...
### STUDENT: anom_dec = ...
### BEGIN SOLUTION
dec_clim = tos.sel(time_counter=tos["time_counter"].dt.month == 12).mean("time_counter")
anom_dec = sst_dec - dec_clim
### END SOLUTION
print(dict(dec_clim.sizes), f"| anomaly from {float(anom_dec.min()):.1f} to {float(anom_dec.max()):.1f} °C")

def draw_box(ax, lon0, lon1, lat0, lat1, color):
    ax.plot([lon0, lon1, lon1, lon0, lon0], [lat0, lat0, lat1, lat1, lat0], color=color, linewidth=2,
            transform=ccrs.PlateCarree())

fig, ax = plt.subplots(figsize=(13, 6.5), subplot_kw={"projection": ccrs.Robinson(central_longitude=-150)})
anom_dec.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.balance, vmin=-3, vmax=3,
              cbar_kwargs={"label": "SST anomaly (°C)", "shrink": 0.7})
draw_box(ax, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX, "magenta")                          # our box
draw_box(ax, NINO_LON_MIN, NINO_LON_MAX, NINO_LAT_MIN, NINO_LAT_MAX, "k")            # Nino 3.4
ax.add_feature(cfeature.LAND, facecolor="#d4c9a8")
ax.coastlines(linewidth=0.5)
ax.set_title("NEMO 1° — SST anomaly, December 2023 minus the mean December 2006–2025")
plt.show()
### HINT: `tos["time_counter"].dt.month == 12` is True for the 20 Decembers: `tos.sel(time_counter=...)` keeps them, `.mean("time_counter")` averages them.
### HINT: Anomaly: `sst_dec - dec_clim` (both are `(lat, lon)` maps).

**✅ Check**: `dec_clim` is `{'lat': 180, 'lon': 360}`, and the anomaly goes from about −4 to +3.5 °C. Look at the map:
- a **warm tongue along the equator**, from the date line (180°) to the coast of South America, **+2 to +3 °C** in the Niño 3.4 box: that is El Niño, **more than 5 000 km** from Costa Rica;
- the warm water **spreads along the American coast**, north to Central America: our box is warmer than usual too (**\~+1.3 °C** on average);
- elsewhere, other anomalies (warm or cold) that have nothing to do with El Niño: the ocean is never "normal" everywhere at once.

**Step 2 — is it a coincidence?** One month proves nothing. With 20 years of model, compare month by month the anomaly in the Niño 3.4 box and in our box.

**🎯 Your turn**: complete the function `box_anomaly`: an **area-weighted mean** over the box, then the **anomaly** relative to the monthly climatology: the two steps of TP2b, Exercises 2-4. The only difference: here the time dimension is `time_counter`, so the group is `"time_counter.month"`.

In [ ]:
def box_anomaly(sst, lon_min, lon_max, lat_min, lat_max):
    """Monthly SST anomaly averaged over a box (longitudes 0-360)."""
    box = sst.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
    weights = np.cos(np.deg2rad(box["lat"]))
    ### STUDENT: # TODO: area-weighted mean over lat and lon -> one value per month (TP2b, Exercise 2)
    ### STUDENT: series = ...
    ### STUDENT: # TODO: minus the monthly climatology (TP2b, Exercises 3 and 4)
    ### STUDENT: clim = ...
    ### STUDENT: return ...
    ### BEGIN SOLUTION
    series = box.weighted(weights).mean(["lat", "lon"])
    clim = series.groupby("time_counter.month").mean()
    return series.groupby("time_counter.month") - clim
    ### END SOLUTION

nino34 = box_anomaly(tos, NINO_LON_MIN, NINO_LON_MAX, NINO_LAT_MIN, NINO_LAT_MAX)
costa_rica = box_anomaly(tos, LON_MIN + 360, LON_MAX + 360, LAT_MIN, LAT_MAX)

dates = nino34["time_counter"].dt.year + (nino34["time_counter"].dt.month - 0.5) / 12   # decimal years (model calendar)
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(dates, nino34, color="k", linewidth=1.5, label="Niño 3.4 (central equatorial Pacific)")
ax.plot(dates, costa_rica, color="magenta", linewidth=1.5, label="our Costa Rica box")
ax.axhline(0, color="0.3", linewidth=0.8)
ax.axhspan(0.5, 4, color="#d6604d", alpha=0.08)                   # above +0.5 °C: El Niño conditions
ax.axhspan(-4, -0.5, color="#4393c3", alpha=0.08)                 # below -0.5 °C: La Niña conditions
ax.set_ylim(-2.5, 3.5)
ax.set_ylabel("SST anomaly (°C)")
ax.set_title("NEMO 1° — monthly SST anomalies, 2006–2025")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.show()

print(f"correlation Nino 3.4 / Costa Rica: {float(xr.corr(nino34, costa_rica)):.2f}")
print(f"standard deviation: Nino 3.4 {float(nino34.std()):.2f} °C | Costa Rica {float(costa_rica.std()):.2f} °C")
### HINT: Weighted mean: `box.weighted(weights).mean(["lat", "lon"])`.
### HINT: Climatology: `series.groupby("time_counter.month").mean()`; anomaly: `series.groupby("time_counter.month") - clim`.

**✅ Check**: correlation **≈ 0.77**; standard deviation **≈ 0.9 °C** in Niño 3.4 and **≈ 0.5 °C** in our box. The two curves go up and down **together**:
- **El Niño** (black above +0.5 °C): 2006-07, 2009-10, **2015-16** (the strongest, **+3.2 °C** in November 2015), **2023-24**. Our box warms too: **\~+1.5 °C** in 2015-16 and 2023-24, less (\~+0.5 °C) in the weaker events;
- **La Niña** (black below −0.5 °C): **2007-08** and **2010-11**: our box is \~1 to 1.5 °C colder than usual. But the long La Niña of **2020-23** barely shows in our box: the link is strong, not automatic.

💬 **Discuss**:
- A large part of the year-to-year variability of SST at Costa Rica is **decided far away**, in the central Pacific. The signal travels along the equator, then along the American coast.
- What would a **regional model of Costa Rica** (TP3b) need to reproduce the warm year 2023? It cannot compute El Niño: its domain is too small. It must **receive** it at its open boundaries, from a global model or a reanalysis. That is the job of a global model in a chain of models.
- A correlation of 0.77 is not 1: the rest of the variability is local (the Papagayo wind, the Costa Rica Dome…).

**🚀 Going further** (optional): is our box **late** compared to Niño 3.4? Compute the correlation when Costa Rica is shifted by 1, 2, 3 months: `xr.corr(nino34[:-lag], costa_rica[lag:].assign_coords(time_counter=nino34["time_counter"][:-lag]))`. <!-- BEGIN SOLUTION -->*(Instructors: 0.77 at lag 0, 0.78 at lag 1, 0.74 at lag 2, 0.66 at lag 3: the peak is at \~1 month.)*<!-- END SOLUTION -->

---
<a id="advanced-a"></a>
## 🚀 Advanced track A — longitudes from −180 to 180

Adding 360 by hand works, but it is easy to forget. A more robust way: convert the longitude **coordinate** of the whole dataset once, then sort it. Do it for `ds2`, call the result `ds2w`, and check that `ds2w["tos"].sel(lon=slice(LON_MIN, LON_MAX))` now works. Be careful: after the conversion, the longitudes must be **sorted**, otherwise `slice` does not work.

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
ds2w = ds2.assign_coords(lon=((ds2["lon"] + 180) % 360) - 180).sortby("lon")
print("lon now from", float(ds2w.lon[0]), "to", float(ds2w.lon[-1]))
box = ds2w["tos"].sel(time_counter="2023-12", lon=slice(LON_MIN, LON_MAX), lat=slice(LAT_MIN, LAT_MAX)).squeeze()
print(dict(box.sizes), "| same values as zoom:", bool((box.values[~np.isnan(box.values)] == zoom.values[~np.isnan(zoom.values)]).all()))
### END SOLUTION
### HINT: `ds2.assign_coords(lon=((ds2["lon"] + 180) % 360) - 180)` maps 0…359 to −180…179 (`%` is the remainder of the division).
### HINT: Then `.sortby("lon")` puts the longitudes back in increasing order.

**✅ Check (advanced)**: longitudes from **−180 to 179**, and the selection with negative longitudes now gives `{'lat': 10, 'lon': 16}`, with exactly the same values as in A.2. Many tools (CDO: `cdo sellonlatbox,-92,-77,4,14`) do the conversion for you; xarray does not.

---
<a id="part-b"></a>
# Part B — z-levels: the vertical grid

<a id="b-1"></a>
## B.1 Where are the model levels?

In NEMO, the ocean is cut into **75 horizontal layers**, at the **same depths everywhere**. `deptht` gives the depth of the **centre** of each layer, `deptht_bnds` its top and bottom.

**🎯 Your turn**: compute the **thickness** of each layer (bottom − top), and count the levels in the upper 100 m and in the upper 1000 m.

In [ ]:
top = ds3["deptht_bnds"].isel(bnds=0)          # top of each layer (m)
bottom = ds3["deptht_bnds"].isel(bnds=1)       # bottom of each layer (m)
### STUDENT: # TODO: thickness of each layer, and number of levels above 100 m and above 1000 m
### STUDENT: thickness = ...
### STUDENT: n100 = ...
### STUDENT: n1000 = ...
### BEGIN SOLUTION
thickness = bottom - top
n100 = int((ds3["deptht"] < 100).sum())
n1000 = int((ds3["deptht"] < 1000).sum())
### END SOLUTION
print(f"levels above 100 m: {n100} | above 1000 m: {n1000} | total: {ds3.sizes['deptht']}")
print(f"thickness: {float(thickness[0]):.1f} m at the surface, {float(thickness[-1]):.0f} m at the bottom")

fig, ax = plt.subplots(figsize=(5, 6))
ax.plot(thickness, ds3["deptht"], "o-", markersize=3)
ax.invert_yaxis()
ax.set_xlabel("Layer thickness (m)")
ax.set_ylabel("Depth (m)")
ax.set_title("NEMO — the 75 z-levels")
plt.show()
### HINT: Thickness: `bottom - top`.
### HINT: `(ds3["deptht"] < 100)` is True/False for each level; `.sum()` counts the True.

**✅ Check**: **\~1 m thick at the surface** and **\~200 m at the bottom**; **24 levels** above 100 m, **46** above 1000 m, out of 75. The levels are **dense near the surface**, where temperature, currents and plankton change fast with depth, and **sparse in the deep ocean**, which is more uniform. This is true **everywhere**: on a shelf 50 m deep, only the levels above 50 m are in the water, all the deeper ones are "in the rock" (NaN).

---
<a id="b-2"></a>
## B.2 A vertical section across the Costa Rica Dome

With z-levels, a section is just a selection: depth is already a coordinate. The model cells are centred on x.5°, so we take the row at **9.5°N**, the closest to the Costa Rica Dome (\~9°N).

**🎯 Your turn**: select the temperature of **December 2006** (the 3D file only has 2006) along **9.5°N**, between 92°W and 84°W. Remember the 0–360 convention!

In [ ]:
LAT_SECTION = 9.5
### STUDENT: # TODO: thetao, last month of the file, latitude 9.5, longitudes 92°W to 84°W (0-360!)
### STUDENT: sec = ds3["thetao"].isel(time_counter=...).sel(lat=..., lon=slice(..., ...))
### BEGIN SOLUTION
sec = ds3["thetao"].isel(time_counter=-1).sel(lat=LAT_SECTION, lon=slice(-92 + 360, -84 + 360))
### END SOLUTION
sec = sec.assign_coords(lon=sec["lon"] - 360)        # back to °E for the axis
print(dict(sec.sizes), "|", sec["time_counter"].values)

fig, ax = plt.subplots(figsize=(12, 5))
sec.plot(ax=ax, x="lon", y="deptht", yincrease=False, cmap="jet", vmin=8, vmax=30,
         cbar_kwargs={"label": "Temperature (°C)"})                      # each rectangle = one model cell
sec.plot.contour(ax=ax, x="lon", y="deptht", yincrease=False, levels=[20], colors="k", linewidths=2)
ax.hlines(ds3["deptht_bnds"].isel(bnds=1), -92.5, -83.5, color="w", linewidth=0.3)   # the z-levels
ax.set_ylim(300, 0)
ax.set_facecolor("#d4c9a8")
ax.set_title("NEMO 1° — temperature along 9.5°N, December 2006 (black: 20 °C, white lines: z-levels)")
plt.show()
### HINT: Last month: `.isel(time_counter=-1)`; latitude: `.sel(lat=LAT_SECTION)` (9.5 exists exactly in the grid).
### HINT: Longitudes: `lon=slice(-92 + 360, -84 + 360)`.

**✅ Check**: `{'deptht': 75, 'lon': 9}`: **9 columns** across 8° of longitude. The section is made of **rectangles**: one per cell (1° wide) and per level. The white lines are **horizontal**: z-levels do not follow the sea floor. The 20 °C isotherm is at **\~40 m** at 92°W, rises to **\~30 m** around **88°W** (the **Costa Rica Dome**), then deepens to **\~50 m** towards the coast. The Dome is there, but drawn with only a few columns.

💬 Zoom out with `ax.set_ylim(4000, 0)` and run again: the deep levels are **\~100-200 m thick**, and the sea floor is a **staircase**, a whole cell (or a part of it) at a time.

---
<a id="b-3"></a>
## B.3 A map at 100 m

On z-levels, a map at a fixed depth is also a simple selection: the nearest level.

**🎯 Your turn**: map the temperature of December 2006 at **100 m**, over the Costa Rica box.

In [ ]:
### STUDENT: # TODO: thetao, last month, the level nearest to 100 m (hint: method="nearest"), the box in 0-360
### STUDENT: t100 = ...
### BEGIN SOLUTION
t100 = ds3["thetao"].isel(time_counter=-1).sel(deptht=100, method="nearest").sel(
    lon=slice(LON_MIN + 360, LON_MAX + 360), lat=slice(LAT_MIN, LAT_MAX))
### END SOLUTION
print(f"Nearest level: {float(t100['deptht']):.1f} m")

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
t100.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="jet", edgecolors="0.5", linewidth=0.3,
          cbar_kwargs={"label": "Temperature (°C)", "shrink": 0.7})
base_map(ax, f"NEMO 1° — temperature at {float(t100['deptht']):.0f} m, December 2006",
         extent=[LON_MIN - 0.5, LON_MAX + 0.5, LAT_MIN - 0.5, LAT_MAX + 0.5])
plt.show()
### HINT: `.sel(deptht=100, method="nearest")`, as for a point with `.sel(lat=..., method="nearest")` in TP1a.
### HINT: Then the same box as in A.2: `lon=slice(LON_MIN + 360, LON_MAX + 360), lat=slice(LAT_MIN, LAT_MAX)`.

**✅ Check**: the nearest level is at **97 m**. On the Pacific side: **\~14-15 °C**, coldest along 6-9°N west of 87°W (the thermocline ridge that includes the Dome). On the Caribbean side: **\~24-26 °C**, the thermocline is much deeper there. A few more cells are blank than at the surface (13 against 7): they are shallower than \~100 m. In z-levels, **a level either is in the water or is not**.

👉 Keep this cell in mind: in TP3b, the same map with CROCO will need a vertical **interpolation**, because its levels have no fixed depth.

---
<a id="b-4"></a>
## B.4 The whole Pacific in one section: the equator

The section of B.2 was 8° long, with 9 columns. A global model can draw the same kind of section **across a whole ocean**. Along the equator (the row at **0.5°N**), from New Guinea (130°E) to the coast of Ecuador (80°W = 280°E): **151 columns, 16 000 km**.

**🎯 Your turn**: select the temperature of December 2006 along the equator, from 130°E to 80°W. In the 0–360 convention the Pacific is **continuous**: one `slice` is enough (try it with −180…180 longitudes: the date line would cut it in two!).

In [ ]:
### STUDENT: # TODO: thetao, last month, latitude 0.5, longitudes 130°E to 80°W (= 280 in 0-360)
### STUDENT: eq = ...
### BEGIN SOLUTION
eq = ds3["thetao"].isel(time_counter=-1).sel(lat=0.5, lon=slice(130, 280))
### END SOLUTION
print(dict(eq.sizes))

fig, ax = plt.subplots(figsize=(14, 4.5))
eq.plot(ax=ax, x="lon", y="deptht", yincrease=False, cmap="jet", vmin=8, vmax=30,
        cbar_kwargs={"label": "Temperature (°C)"})
eq.plot.contour(ax=ax, x="lon", y="deptht", yincrease=False, levels=[20], colors="k", linewidths=2)
ax.axvspan(268, 280, color="magenta", alpha=0.15)                 # the longitudes of our box
ax.set_ylim(400, 0)
ax.set_facecolor("#d4c9a8")
ax.set_xlabel("Longitude (°E, 0-360)")
ax.set_title("NEMO 1° — temperature along the equator, December 2006 (black: 20 °C, magenta: longitudes of our box)")
plt.show()

z20 = eq["deptht"].where(eq < 20).min("deptht")      # depth of the first level colder than 20 °C
print(f"20 °C at ~{float(z20.sel(lon=150)):.0f} m at 150°E, ~{float(z20.sel(lon=265)):.0f} m at 95°W")
### HINT: Same as B.2: `ds3["thetao"].isel(time_counter=-1).sel(lat=0.5, lon=slice(130, 280))`.

**✅ Check**: `{'deptht': 75, 'lon': 151}`. The 20 °C isotherm is at **\~180 m** in the west (150°E) and rises to **\~60 m** in the east (95°W): the **thermocline slopes** across the whole Pacific. The trade winds push the warm surface water to the west, where it piles up (the warm pool of A.1); in the east, cold water stays close to the surface. The shallow thermocline of the **Costa Rica Dome** (\~30 m in B.2) is the northern end of this eastern shallow side.

💬 During El Niño the trade winds weaken and the slope **relaxes**: the warm water of the west flows back east, the thermocline deepens in the east, and the surface warms (A.3). This is the mechanism behind the correlation of A.3, and it happens on a scale of **10 000 km**: a regional model of Costa Rica cannot compute it, it must **receive** it at its open boundaries (TP3b).

---
<a id="advanced-b"></a>
## 🚀 Advanced track B — the depth of the thermocline over the whole tropical Pacific

B.4 was one line. Make it a **map**: for each cell of the tropical Pacific (20°S–20°N, 120°E–70°W), the depth of the 20 °C isotherm in December 2006 (the same `z20` trick as at the end of B.4). Where is the thermocline deepest? Find the **Costa Rica Dome** on your map: how does it stand out from its surroundings?

In [ ]:
### STUDENT: # 🚀 Your code here
### BEGIN SOLUTION
t = ds3["thetao"].isel(time_counter=-1).sel(lat=slice(-20, 20), lon=slice(120, 290))
z20_map = t["deptht"].where(t < 20).min("deptht").where(t.isel(deptht=0).notnull())   # mask the land

fig, ax = plt.subplots(figsize=(14, 5), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
z20_map.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.deep, vmin=0, vmax=250,
             cbar_kwargs={"label": "Depth of 20 °C (m)", "shrink": 0.8})
z20_map.plot.contour(ax=ax, transform=ccrs.PlateCarree(), levels=[50], colors="w", linewidths=1)
ax.plot([LON_MIN, LON_MAX, LON_MAX, LON_MIN, LON_MIN], [LAT_MIN, LAT_MIN, LAT_MAX, LAT_MAX, LAT_MIN],
        color="magenta", linewidth=2, transform=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#d4c9a8")
ax.coastlines(linewidth=0.5)
ax.set_title("NEMO 1° — depth of the 20 °C isotherm, December 2006 (white: 50 m)")
plt.show()
print(f"z20: {float(z20_map.sel(lat=15.5, lon=150)):.0f} m at 15.5°N 150°E, {float(z20_map.sel(lat=0.5, lon=265)):.0f} m at the equator 95°W, "
      f"{float(z20_map.sel(lat=9.5, lon=272)):.0f} m in the Dome (9.5°N, 88°W)")
### END SOLUTION
### HINT: Select a 3D block `t = ds3["thetao"].isel(time_counter=-1).sel(lat=slice(-20, 20), lon=slice(120, 290))`, then `t["deptht"].where(t < 20).min("deptht")`.
### HINT: Land cells give no value colder than 20 °C: mask them with `.where(t.isel(deptht=0).notnull())`. For the map, `ccrs.PlateCarree(central_longitude=180)` keeps the Pacific in one piece.

**✅ Check (advanced)**: the thermocline is **deepest (> 200 m) in the western Pacific**, off the equator (\~15°N and \~15°S, e.g. \~240 m at 15.5°N 150°E), and **shallowest (< 50 m) in the east**: along the equator (\~60 m at 95°W), along Peru, and in a **ridge along \~9°N** that ends in the **Costa Rica Dome** (\~30 m). In our box (magenta), the contrast is striking: \~30 m on the Pacific side, **more than 200 m** on the Caribbean side. The Dome is not a local accident: it is the eastern end of a structure that crosses the basin, driven by the winds over the whole Pacific.

---
<a id="c-debugging"></a>
## C. 🐞 Debugging exercise

A colleague used to CROCO wants the NEMO temperature at 100 m around Costa Rica, in December 2006. *"First it crashed, then I got an empty plot!"* The cell contains **three bugs**.

Run it, read the error message (the last line), fix, run again… until you get the map of B.3.

In [ ]:
### STUDENT: # 🐞 Fix me! (3 bugs)
### STUDENT: t_bug = ds3["temp"].isel(time_counter=-1).sel(depth=100, method="nearest")
### STUDENT: t_bug = t_bug.sel(lon=slice(LON_MIN, LON_MAX), lat=slice(LAT_MIN, LAT_MAX))
### STUDENT: print(dict(t_bug.sizes))
### BEGIN SOLUTION
# Fixed version
t_bug = ds3["thetao"].isel(time_counter=-1).sel(deptht=100, method="nearest")
t_bug = t_bug.sel(lon=slice(LON_MIN + 360, LON_MAX + 360), lat=slice(LAT_MIN, LAT_MAX))
print(dict(t_bug.sizes))
### END SOLUTION

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
t_bug.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="jet")
base_map(ax, "NEMO 1° — temperature at 100 m, December 2006")
plt.show()
### BEGIN SOLUTION

# Bug 1: KeyError 'temp'            -> in NEMO the temperature is thetao (temp is the CROCO name).
# Bug 2: 'depth' is not a dimension -> the vertical dimension is deptht.
# Bug 3 (silent): lon=slice(-92, -77) is empty in a 0-360 file: {'lon': 0}, then the plot fails
#        ("TypeError: No numeric data to plot"). The error appears in the plot, but the bug is in the selection.
### END SOLUTION
### HINT: Read the last line of each error. List the variables with `list(ds3.data_vars)` and the dimensions with `ds3["thetao"].dims`.
### HINT: When the plot fails, look at the `print` just before it: how many longitudes are left? (section A.2)

**✅ Check**: `{'lat': 10, 'lon': 16}` and the map of B.3. Note the third bug: the error message comes from the **plot**, but the bug is two lines higher, in the **selection**. When a plot fails, first print the sizes of what you are plotting.

---
<a id="d-recap"></a>
## D. Recap: what to remember for TP3b

Complete the NEMO column with your partner (double-click on this cell). You will fill in the CROCO column at the end of TP3b, right after.

| | NEMO (TP3a) | CROCO (TP3b) |
|---|---|---|
| Domain | | |
| Horizontal grid (spacing, 1D or 2D coordinates) | | |
| Longitude convention | | |
| Vertical levels: name, number, do they follow the sea floor? | | |
| Temperature at 100 m: how? | | |
| Shelf and coastline | | |
| What is needed at the open boundaries | | |
| Can it compute El Niño by itself? | | |

<!-- SOLUTION ONLY -->
### 👩‍🏫 Instructor notes

Expected NEMO column: global · 1° (\~110 km), regular grid, 1D `lon`/`lat` (regridded from ORCA1) · 0 to 360 · z-levels: `deptht`, 75 levels, fixed depths, do not follow the sea floor (staircase) · `.sel(deptht=100, method="nearest")` · shelf and small gulfs smaller than one cell, staircase coast · nothing: no open boundary (only the atmosphere at the surface) · yes: El Niño is born in the model itself (A.3, B.4).

Expected CROCO column (TP3b): North-East Tropical Pacific · \~5.5 km, curvilinear grid, 2D `lon_rho`/`lat_rho` · −180 to 180 · sigma levels: `s_rho`, 50 levels, follow the sea floor, depth to compute · vertical interpolation (`at_depth`) · the shelf has its 50 levels · the ocean state at the open boundaries (temperature, salinity, currents, sea level), from a global model or reanalysis · no: its domain is too small, El Niño enters through its open boundaries.

TP3a (\~2h) and TP3b (\~1h) fill the same morning (9:00–12:00): take the break between the two. Timing of TP3a: section 1 ≈ 20 min (discuss the anatomy with the whole group), Part A ≈ 40 min (A.3 is the key message of the session: "the global model sees where it comes from"; take 5 min to discuss its plots with the whole group), Part B ≈ 35 min, C + D ≈ 15 min, + \~10 min of margin. If short of time, skip B.3 rather than A.3 or B.4. The NEMO grid is regular: the code is the same as in TP1/TP2, the effort goes into reading the figures.

About the cells "on land" in A.2: the outputs were regridded from ORCA1 to 1° with `cdo remapdis` (distance-weighted, see the `history` attribute), which gives values to target cells partly over land. They are not ocean in the model itself.